# 07_01 — Counterfactual Backtest: All Strategies Over the Validation Period

## 1. Objective and Temporal Framework

This notebook runs a **counterfactual backtest** — the core evaluation of the DSS.
We simulate what total energy procurement cost would have been under each strategy,
applied to the same historical validation period with the same market prices.

### Strategies compared
| Strategy | Description |
|---|---|
| `spot_only` | Buy 100% at daily spot price — no hedging, no flexibility |
| `static_hedge` | Pre-buy 70% at front-month futures; remainder at spot |
| `heuristic_policy` | DSS rule engine — hedge or shift only when tail-risk signals fire |
| `rl_policy` | DSS RL engine — actions learned from reward signal |

### Temporal causality guarantee
The OMIP Day-Ahead auction publishes the t+1 spot price at **13:00 on day t**.
This means:
- **t+1 price is KNOWN** at decision time — treated as realized market input.
- **t+2, t+3 and beyond are UNKNOWN** — the DSS uses quantile predictions (q50, q90).
- All lag and rolling features use `pandas.shift()` — strictly past information.
- Train / validation splits are **chronological** with no row overlap.

No look-ahead bias is introduced at any stage of the pipeline.

**Source data:** `data/processed/train_features.csv` · `data/processed/validation_features.csv`

**Core modules:** `src/backtesting/simulate_baseline.py` · `simulate_policy.py` · `simulate_rl_policy.py`

In [16]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df      = pd.read_csv(project_root / 'data/processed/train_features.csv')
validation_df = pd.read_csv(project_root / 'data/processed/validation_features.csv')

print(f'Train:      {train_df.shape[0]:,} rows  |  {train_df["date"].min()} -> {train_df["date"].max()}')
print(f'Validation: {validation_df.shape[0]:,} rows  |  {validation_df["date"].min()} -> {validation_df["date"].max()}')

Train:      1,461 rows  |  2020-01-01 -> 2023-12-31
Validation: 366 rows  |  2024-01-01 -> 2024-12-31


## 2. Build Policy Inputs

The `prepare_policy_inputs()` bridge merges quantile forecasts (trained on `train_df`,
evaluated on `validation_df`) with the market variables needed by both decision engines.
Columns `q_0.5` and `q_0.9` are **forecast inputs** to the policy — not realized costs.

In [17]:
from src.models.train_model import train_quantile_suite
from src.decision.policy_inputs import prepare_policy_inputs, summarize_policy_inputs

quantile_output  = train_quantile_suite(train_df=train_df, eval_df=validation_df)
policy_inputs_df = prepare_policy_inputs(validation_df, quantile_output.results)

print('Policy inputs overview:')
display(summarize_policy_inputs(policy_inputs_df))
display(policy_inputs_df[['date', 'Spot_Price_SPEL', 'Future_M1_Price', 'q_0.5', 'q_0.9']].head(8))

Policy inputs overview:


,n_rows,date_min,date_max,n_quantile_columns,has_weekend_flag,has_holiday_flag,has_volume_column
0,337,2024-01-29,2024-12-30,3,True,False,False


,date,Spot_Price_SPEL,Future_M1_Price,q_0.5,q_0.9
0,2024-01-29,77.97,60.26,80.235959,83.679672
1,2024-01-30,81.28,59.30,86.721442,84.432614
2,2024-01-31,84.26,61.65,83.379503,84.115509
3,2024-02-01,70.95,48.20,71.289227,77.015751
4,2024-02-02,65.32,48.00,61.630357,73.029693
5,2024-02-03,60.32,48.00,53.123370,60.899687
6,2024-02-04,63.10,48.00,62.846819,68.786243
7,2024-02-05,80.92,44.88,64.322989,77.105289


## 3. Simulate All Four Strategies

Each simulator converts daily procurement decisions into **counterfactual cost** rows.
The realized `Spot_Price_SPEL` is used as the cost basis (t+1 price, known at decision time).

### Factory energy model (from `src/config/constants.py`)
```
energy_consumed = base_load (0.30) + variable_load (0.70) x production_level
total_cost      = energy_consumed x unit_price
```
- `do_nothing`       -> unit_price = spot
- `buy_m1_future`    -> unit_price = futures (locked in advance)
- `shift_production` -> reduced load + shift penalty per MWh

In [18]:
from src.backtesting.simulate_baseline import (
    simulate_spot_only_baseline,
    simulate_static_hedge_baseline,
)
from src.backtesting.simulate_policy import simulate_policy_strategy
from src.backtesting.simulate_rl_policy import simulate_rl_policy_strategy
from src.decision.heuristic_policy import apply_heuristic_policy
from src.rl.train_rl_agent import train_q_learning_agent
from src.decision.rl_policy import apply_rl_policy

spot_only_df    = simulate_spot_only_baseline(policy_inputs_df)
static_hedge_df = simulate_static_hedge_baseline(policy_inputs_df)

heuristic_policy_df = apply_heuristic_policy(policy_inputs_df)
heuristic_sim_df    = simulate_policy_strategy(heuristic_policy_df)

rl_artifacts        = train_q_learning_agent(policy_inputs_df)
rl_policy_artifacts = apply_rl_policy(agent=rl_artifacts.agent, policy_inputs_df=policy_inputs_df)

# decisions_df only contains date + action columns; merge price data before simulation
rl_decisions_with_prices = rl_policy_artifacts.decisions_df.merge(
    policy_inputs_df[["date", "Spot_Price_SPEL", "Future_M1_Price"]],
    on="date", how="left",
)
rl_sim_df = simulate_rl_policy_strategy(rl_decisions_with_prices)

print('All four strategies simulated.')
for name, df in [('spot_only', spot_only_df), ('static_hedge', static_hedge_df),
                  ('heuristic', heuristic_sim_df), ('rl_policy', rl_sim_df)]:
    print(f'  {name:15s}: {len(df)} rows  |  total_cost = {df["total_cost"].sum():.2f}')

2026-04-26 23:21:10 | INFO | src.rl.train_rl_agent | Starting RL training...
2026-04-26 23:21:10 | INFO | src.rl.train_rl_agent | Training dataframe shape: (337, 13)
2026-04-26 23:21:10 | INFO | src.rl.train_rl_agent | Episodes: 1000
2026-04-26 23:21:36 | INFO | src.rl.train_rl_agent | RL training completed successfully.
2026-04-26 23:21:36 | INFO | src.rl.train_rl_agent | Q-table states learned: 268
2026-04-26 23:21:36 | INFO | src.rl.train_rl_agent | Last episode reward: -11309.2550
2026-04-26 23:21:36 | INFO | src.rl.train_rl_agent | Best episode reward: -10969.1050
2026-04-26 23:21:36 | INFO | src.rl.train_rl_agent | Episode steps summary | mean=337.00 | last=337.00
2026-04-26 23:21:36 | INFO | src.rl.train_rl_agent | Reward summary:
   n_episodes   reward_mean   reward_std  reward_min  reward_max  reward_last  \
0        1000 -12859.089095  1467.036814  -19528.695  -10969.105   -11309.255   

   reward_rolling_mean_last_10  steps_mean  steps_last  q_table_states  \
0              

All four strategies simulated.
  spot_only      : 337 rows  |  total_cost = 20880.38
  static_hedge   : 337 rows  |  total_cost = 21679.82
  heuristic      : 337 rows  |  total_cost = 19231.85
  rl_policy      : 337 rows  |  total_cost = 1759.64


## 4. Daily Cost Table

The combined view shows all four cost columns aligned by date.
Differences on individual days reveal the economic impact of each decision.

In [19]:
from src.backtesting.compare_strategies import compare_daily_costs

daily_costs_df = compare_daily_costs([spot_only_df, static_hedge_df, heuristic_sim_df, rl_sim_df])

display(daily_costs_df.head(12).round(2))
print(f'Date range: {daily_costs_df["date"].min()} -> {daily_costs_df["date"].max()}')

/var/folders/0f/zds9whdd02q1swd8qb96m5m00000gn/T/ipykernel_36702/136322963.py:5: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(daily_costs_df.head(12).round(2))


,date,total_cost_spot_only,action_spot_only,total_cost_static_hedge,action_static_hedge,total_cost_heuristic_policy,action_heuristic_policy,total_cost_rl_policy,action_rl_policy
0,2024-01-29,77.97,buy_on_spot,65.57,static_m1_hedge,60.26,buy_m1_future,2.00,shift_production
1,2024-01-30,81.28,buy_on_spot,65.89,static_m1_hedge,59.30,buy_m1_future,2.00,shift_production
2,2024-01-31,84.26,buy_on_spot,68.43,static_m1_hedge,61.65,buy_m1_future,2.00,shift_production
3,2024-02-01,70.95,buy_on_spot,55.03,static_m1_hedge,48.20,buy_m1_future,2.00,shift_production
4,2024-02-02,65.32,buy_on_spot,53.20,static_m1_hedge,48.00,buy_m1_future,2.00,shift_production
5,2024-02-03,60.32,buy_on_spot,51.70,static_m1_hedge,48.00,buy_m1_future,2.00,shift_production
6,2024-02-04,63.10,buy_on_spot,52.53,static_m1_hedge,48.00,buy_m1_future,63.10,do_nothing
7,2024-02-05,80.92,buy_on_spot,55.69,static_m1_hedge,44.88,buy_m1_future,2.00,shift_production
8,2024-02-06,74.03,buy_on_spot,54.41,static_m1_hedge,46.00,buy_m1_future,74.03,do_nothing
9,2024-02-07,50.11,buy_on_spot,45.66,static_m1_hedge,43.75,buy_m1_future,2.00,shift_production


Date range: 2024-01-29 00:00:00 -> 2024-12-30 00:00:00


## 5. Aggregate Summary

Key metrics across the full validation period:
- **total_cost** — sum of daily procurement costs (lower is better)
- **average_unit_cost** — EUR per MWh procured (normalised by volume)
- **daily_cost_volatility** — std dev of daily cost (stability proxy; lower is better)
- **max_daily_cost** — worst-case single day (tail-risk exposure)
- **savings_vs_spot_only** — absolute savings compared to the do-nothing baseline
- **savings_share_vs_spot_only** — savings as a fraction of spot-only cost

In [20]:
from src.backtesting.compare_strategies import build_strategy_comparison_report

report = build_strategy_comparison_report(
    [spot_only_df, static_hedge_df, heuristic_sim_df, rl_sim_df],
    reference_strategy_name='spot_only',
)

print('Full summary (sorted by total cost):')
display(report['summary_table'].round(4))

print('\nSavings vs spot-only:')
cols = ['strategy_name', 'total_cost', 'savings_vs_spot_only',
        'savings_share_vs_spot_only', 'daily_cost_volatility']
available = [c for c in cols if c in report['summary_vs_reference'].columns]
display(report['summary_vs_reference'][available].round(4))

Full summary (sorted by total cost):


,strategy_name,n_days,total_volume_mwh,total_cost,average_unit_cost,daily_cost_volatility,max_daily_cost,min_daily_cost,n_buy_on_spot_days,n_static_m1_hedge_days,n_buy_m1_future_days,n_do_nothing_days,n_shift_production_days
0,rl_policy,337,337.0,1759.640,5.2215,14.3838,140.94,0.440,NaN,NaN,9.0,40.0,288.0
1,heuristic_policy,337,337.0,19231.850,57.0678,31.2159,133.60,1.670,NaN,NaN,240.0,65.0,32.0
2,spot_only,337,337.0,20880.380,61.9596,39.6302,146.67,0.440,337.0,NaN,NaN,NaN,NaN
3,static_hedge,337,337.0,21679.815,64.3318,27.0975,113.12,16.678,NaN,337.0,NaN,NaN,NaN



Savings vs spot-only:


,strategy_name,total_cost,savings_vs_spot_only,savings_share_vs_spot_only,daily_cost_volatility
0,rl_policy,1759.640,19120.740,0.9157,14.3838
1,heuristic_policy,19231.850,1648.530,0.0790,31.2159
2,spot_only,20880.380,0.000,0.0000,39.6302
3,static_hedge,21679.815,-799.435,-0.0383,27.0975


## 6. Heuristic Policy Action Distribution

High `do_nothing` frequency is expected — the heuristic engine hedges only when the
tail-risk premium over the futures price exceeds the configured threshold.
This selectivity is a feature, not a bug: it avoids unnecessary transaction costs.

In [21]:
from src.decision.heuristic_policy import summarize_policy_actions

print('Heuristic policy action mix:')
display(summarize_policy_actions(heuristic_policy_df))

print('\nSample of heuristic decisions:')
sample_cols = ['date', 'recommended_action', 'decision_reason', 'tail_vs_future_abs']
avail = [c for c in sample_cols if c in heuristic_policy_df.columns]
display(heuristic_policy_df[avail].head(10))

Heuristic policy action mix:


,recommended_action,n_days,share
0,buy_m1_future,240,0.712166
1,do_nothing,65,0.192878
2,shift_production,32,0.094955



Sample of heuristic decisions:


,date,recommended_action,decision_reason,tail_vs_future_abs
0,2024-01-29,buy_m1_future,Tail risk exceeds futures price threshold,23.419672
1,2024-01-30,buy_m1_future,Tail risk exceeds futures price threshold,25.132614
2,2024-01-31,buy_m1_future,Tail risk exceeds futures price threshold,22.465509
3,2024-02-01,buy_m1_future,Tail risk exceeds futures price threshold,28.815751
4,2024-02-02,buy_m1_future,Tail risk exceeds futures price threshold,25.029693
5,2024-02-03,buy_m1_future,Tail risk exceeds futures price threshold,12.899687
6,2024-02-04,buy_m1_future,Tail risk exceeds futures price threshold,20.786243
7,2024-02-05,buy_m1_future,Tail risk exceeds futures price threshold,32.225289
8,2024-02-06,buy_m1_future,Tail risk exceeds futures price threshold,29.885150
9,2024-02-07,buy_m1_future,Tail risk exceeds futures price threshold,18.247292
